In [1]:
# CELL 1: Imports
import os
import sys
import json
import time
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.models.factory import create_model
from src.dataset import get_dataloaders
from src.losses import FollicleAwareFocalLoss
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_roc_curves, CLASS_NAMES
from src.gradcam import GradCAM, overlay_cam_on_image

print(f"[PyTorch Version] {torch.__version__}")

[PyTorch Version] 2.13.0+cpu


In [2]:
# CELL 2: Configuration
MODEL_NAME = "ResNet50"
NUM_CLASSES = 3
BATCH_SIZE = 16
LEARNING_RATE = 0.0001
EPOCHS = 2
CHECKPOINT_PATH = "results/checkpoints/ResNet50_best.pth"
MANIFEST_PATH = "manifest.csv" if os.path.exists("manifest.csv") else "results/manifest.csv"
OUTPUT_DIR = Path("results") / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configured {MODEL_NAME}:")
print(f"  Learning Rate:   {LEARNING_RATE}")
print(f"  Batch Size:      {BATCH_SIZE}")
print(f"  Checkpoint Path: {CHECKPOINT_PATH} (Exists: {os.path.exists(CHECKPOINT_PATH)})")

Configured ResNet50:
  Learning Rate:   0.0001
  Batch Size:      16
  Checkpoint Path: results/checkpoints/ResNet50_best.pth (Exists: False)


In [3]:
# CELL 3: Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Deterministic seed set to {SEED}")

Deterministic seed set to 42


In [4]:
# CELL 4: GPU verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware compute device: {device}")
if device.type == "cuda":
    print(f"  Device Name: {torch.cuda.get_device_name(0)}")
    print(f"  Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
else:
    print("  Running on CPU execution mode.")

Hardware compute device: cpu
  Running on CPU execution mode.


In [5]:
# CELL 5: Dataset loading
df = pd.read_csv(MANIFEST_PATH)
print(f"Master manifest loaded from {MANIFEST_PATH}")
print(f"Total scans: {len(df)}")
for split in ["train", "val", "test"]:
    print(f"  Split '{split:5s}': {(df['split'] == split).sum()} scans")

Master manifest loaded from manifest.csv
Total scans: 571
  Split 'train': 400 scans
  Split 'val  ': 85 scans
  Split 'test ': 86 scans


In [6]:
# CELL 6: Class distribution
ct = pd.crosstab(df["split"], df["class_name"])[["Normal Ovary", "PCOS", "Dominant Follicle"]]
print("Class breakdown across dataset splits:")
print(ct)

Class breakdown across dataset splits:
class_name  Normal Ovary  PCOS  Dominant Follicle
split                                            
test                  34    21                 31
train                155    97                148
val                   32    21                 32


In [7]:
# CELL 7: DataLoader
train_loader, val_loader, test_loader, class_weights = get_dataloaders(
    manifest_path=MANIFEST_PATH,
    batch_size=BATCH_SIZE,
    num_workers=0,
    use_sampler=True
)
print(f"DataLoaders created:")
print(f"  Train: {len(train_loader)} batches | Val: {len(val_loader)} batches | Test: {len(test_loader)} batches")
print(f"  Class Weights (inverse frequency): {class_weights.tolist()}")

DataLoaders created:
  Train: 25 batches | Val: 6 batches | Test: 6 batches
  Class Weights (inverse frequency): [0.8602150678634644, 1.3745704889297485, 0.9009009003639221]


In [8]:
# CELL 8: Model definition
# Check if checkpoint exists
load_path = CHECKPOINT_PATH if os.path.exists(CHECKPOINT_PATH) else None
model, target_layer_name = create_model(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES,
    pretrained=True,
    checkpoint_path=load_path
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Instantiated {MODEL_NAME}:")
print(f"  Total Parameters:     {total_params:,}")
print(f"  Trainable Parameters: {trainable_params:,}")
print(f"  Target Layer for XAI: {target_layer_name}")
print(f"  Initial Checkpoint:   {'LOADED FROM ' + load_path if load_path else 'ImageNet Pretrained Initialization'}")

Instantiated ResNet50:
  Total Parameters:     23,514,179
  Trainable Parameters: 23,514,179
  Target Layer for XAI: layer4
  Initial Checkpoint:   ImageNet Pretrained Initialization


In [9]:
# CELL 9: Train function
def train_single_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets, _ in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
        
    return running_loss / max(total, 1), correct / max(total, 1)

In [10]:
# CELL 10: Validation function
def run_validation(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0
    with torch.no_grad():
        for inputs, targets, _ in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * inputs.size(0)
            total += targets.size(0)
    val_loss = running_loss / max(total, 1)
    
    metrics, cm, probs, preds = evaluate_model(model, loader, device)
    metrics["Val_Loss"] = round(val_loss, 4)
    return metrics, cm, probs, preds

In [11]:
# CELL 11: Training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)

# Use FollicleAwareFocalLoss for ConvNeXt-Tiny V6, CrossEntropy for baseline CNNs/Transformers
if "ConvNeXt" in MODEL_NAME:
    criterion = FollicleAwareFocalLoss(gamma=2.0)
    print("Using custom FollicleAwareFocalLoss for ConvNeXt-Tiny V6")
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    print(f"Using CrossEntropyLoss with class weights: {class_weights.tolist()}")

history = []
best_val_macro_f1 = 0.0
best_epoch = 0

# If checkpoint already loaded, evaluate baseline immediately
val_metrics, cm, probs, preds = run_validation(model, val_loader, criterion, device)
best_val_macro_f1 = val_metrics["Macro_F1"]
print(f"[Initial Validation] Macro-F1: {best_val_macro_f1:.4f} | Accuracy: {val_metrics['Accuracy']:.4f}")

if not os.path.exists(CHECKPOINT_PATH):
    print(f"Checkpoint not found. Executing {EPOCHS} fine-tuning epochs...")
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_single_epoch(model, train_loader, optimizer, criterion, device)
        v_metrics, v_cm, v_probs, v_preds = run_validation(model, val_loader, criterion, device)
        elapsed = time.time() - t0
        
        v_f1 = v_metrics["Macro_F1"]
        history.append({
            "epoch": epoch,
            "train_loss": round(tr_loss, 4),
            "train_acc": round(tr_acc, 4),
            "val_loss": v_metrics["Val_Loss"],
            "val_acc": v_metrics["Accuracy"],
            "val_macro_f1": v_f1,
            "val_macro_auc": v_metrics["Macro_AUC"],
            "time_sec": round(elapsed, 2)
        })
        
        print(f"Epoch [{epoch}/{EPOCHS}] Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Val Loss: {v_metrics['Val_Loss']:.4f} F1: {v_f1:.4f} AUC: {v_metrics['Macro_AUC']:.4f} ({elapsed:.1f}s)")
        
        if v_f1 >= best_val_macro_f1:
            best_val_macro_f1 = v_f1
            best_epoch = epoch
            torch.save(model.state_dict(), CHECKPOINT_PATH)
            val_metrics, cm, probs, preds = v_metrics, v_cm, v_probs, v_preds
            print(f"  --> Saved new best checkpoint to {CHECKPOINT_PATH} (Macro-F1: {best_val_macro_f1:.4f})")
else:
    print(f"Using pre-existing verified best checkpoint: {CHECKPOINT_PATH}")
    best_epoch = 2

Using CrossEntropyLoss with class weights: [0.8602150678634644, 1.3745704889297485, 0.9009009003639221]
[Initial Validation] Macro-F1: 0.2894 | Accuracy: 0.3529
Checkpoint not found. Executing 2 fine-tuning epochs...
Epoch [1/2] Train Loss: 0.8308 Acc: 0.5950 | Val Loss: 1.0401 F1: 0.3178 AUC: 0.8305 (98.1s)
  --> Saved new best checkpoint to results/checkpoints/ResNet50_best.pth (Macro-F1: 0.3178)
Epoch [2/2] Train Loss: 0.5703 Acc: 0.7975 | Val Loss: 0.6987 F1: 0.7480 AUC: 0.9164 (98.5s)
  --> Saved new best checkpoint to results/checkpoints/ResNet50_best.pth (Macro-F1: 0.7480)


In [12]:
# CELL 12: Best checkpoint selection
print("=" * 60)
print(f"BEST CHECKPOINT SELECTION: {MODEL_NAME}")
print("=" * 60)
print(f"Selection Metric:       Validation Macro-F1")
print(f"Best Validation Epoch:  {best_epoch}")
print(f"Best Macro-F1:          {best_val_macro_f1:.4f}")
print(f"Active Checkpoint Path: {CHECKPOINT_PATH}")

BEST CHECKPOINT SELECTION: ResNet50
Selection Metric:       Validation Macro-F1
Best Validation Epoch:  2
Best Macro-F1:          0.7480
Active Checkpoint Path: results/checkpoints/ResNet50_best.pth


In [13]:
# CELL 13: Validation metrics
print("=" * 60)
print(f"VALIDATION PERFORMANCE METRICS: {MODEL_NAME}")
print("=" * 60)
metrics_table = pd.DataFrame([
    {"Metric": "Accuracy", "Value": f"{val_metrics['Accuracy']*100:.2f}%"},
    {"Metric": "Balanced Accuracy", "Value": f"{val_metrics['Balanced_Accuracy']*100:.2f}%"},
    {"Metric": "Macro Precision", "Value": f"{val_metrics['Macro_Precision']*100:.2f}%"},
    {"Metric": "Macro Recall", "Value": f"{val_metrics['Macro_Recall']*100:.2f}%"},
    {"Metric": "Macro F1-Score", "Value": f"{val_metrics['Macro_F1']*100:.2f}%"},
    {"Metric": "Weighted F1-Score", "Value": f"{val_metrics['Weighted_F1']*100:.2f}%"},
    {"Metric": "Macro ROC-AUC", "Value": f"{val_metrics['Macro_AUC']:.4f}"},
    {"Metric": "Inference Latency", "Value": f"{val_metrics['Inference_Latency_ms']:.2f} ms/scan"},
    {"Metric": "Parameter Count", "Value": f"{val_metrics['Parameters']:,}"}
])
print(metrics_table.to_string(index=False))

VALIDATION PERFORMANCE METRICS: ResNet50
           Metric         Value
         Accuracy        74.12%
Balanced Accuracy        75.45%
  Macro Precision        77.46%
     Macro Recall        75.45%
   Macro F1-Score        74.80%
Weighted F1-Score        74.57%
    Macro ROC-AUC        0.9164
Inference Latency 57.41 ms/scan
  Parameter Count    23,514,179


In [14]:
# CELL 14: Confusion matrix
cm_save_path = str(OUTPUT_DIR / "confusion_matrix.png")
plot_confusion_matrix(cm, MODEL_NAME, cm_save_path)
print(f"✓ Saved Confusion Matrix plot to: {cm_save_path}")

print("\nConfusion Matrix Values (Rows: Ground Truth, Cols: Prediction):")
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
print(cm_df)

✓ Saved Confusion Matrix plot to: results\ResNet50\confusion_matrix.png

Confusion Matrix Values (Rows: Ground Truth, Cols: Prediction):
                   Normal Ovary  PCOS  Dominant Follicle
Normal Ovary                 21     0                 11
PCOS                          0    18                  3
Dominant Follicle             0     8                 24


In [15]:
# CELL 15: ROC-AUC
targets_val = np.array(val_loader.dataset.targets)
roc_save_path = str(OUTPUT_DIR / "roc_curves.png")
plot_roc_curves(targets_val, probs, MODEL_NAME, roc_save_path)
print(f"✓ Saved ROC Curves plot to: {roc_save_path}")
print(f"Macro ROC-AUC: {val_metrics['Macro_AUC']:.4f}")

✓ Saved ROC Curves plot to: results\ResNet50\roc_curves.png
Macro ROC-AUC: 0.9164


In [16]:
# CELL 16: Per-class metrics
per_class_df = pd.DataFrame([
    {
        "Class": "Normal Ovary",
        "Precision": val_metrics["Normal_Precision"],
        "Recall": val_metrics["Normal_Recall"],
        "F1-Score": val_metrics["Normal_F1"]
    },
    {
        "Class": "PCOS",
        "Precision": val_metrics["PCOS_Precision"],
        "Recall": val_metrics["PCOS_Recall"],
        "F1-Score": val_metrics["PCOS_F1"]
    },
    {
        "Class": "Dominant Follicle",
        "Precision": val_metrics["DF_Precision"],
        "Recall": val_metrics["DF_Recall"],
        "F1-Score": val_metrics["DF_F1"]
    }
])
print("Per-Class Diagnostic Performance:")
print(per_class_df.to_string(index=False))

Per-Class Diagnostic Performance:
            Class  Precision  Recall  F1-Score
     Normal Ovary     1.0000  0.6562    0.7925
             PCOS     0.6923  0.8571    0.7660
Dominant Follicle     0.6316  0.7500    0.6857


In [17]:
# CELL 17: Error analysis
print("=" * 60)
print("CLINICAL WEAK-CLASS & CROSS-FOLLICULAR ERROR ANALYSIS")
print("=" * 60)
print(f"PCOS -> Dominant Follicle False Negatives: {val_metrics['PCOS_to_DF']}")
print(f"Dominant Follicle -> PCOS False Positives: {val_metrics['DF_to_PCOS']}")
print(f"Normal -> PCOS Overdiagnosis:               {val_metrics['Normal_to_PCOS']}")
print(f"PCOS -> Normal Underdiagnosis:              {val_metrics['PCOS_to_Normal']}")

# Detailed examination of misclassified cases
val_df = pd.DataFrame({
    "patient_id": [Path(p).stem for p in val_loader.dataset.paths],
    "true_label": [CLASS_NAMES[t] for t in targets_val],
    "pred_label": [CLASS_NAMES[p] for p in preds],
    "pcos_prob": np.round(probs[:, 1], 4),
    "df_prob": np.round(probs[:, 2], 4),
    "normal_prob": np.round(probs[:, 0], 4)
})
misclassified = val_df[val_df["true_label"] != val_df["pred_label"]]
print(f"\nTotal Validation Misclassifications: {len(misclassified)} / {len(val_df)}")
print(misclassified.head(10).to_string(index=False))

CLINICAL WEAK-CLASS & CROSS-FOLLICULAR ERROR ANALYSIS
PCOS -> Dominant Follicle False Negatives: 3
Dominant Follicle -> PCOS False Positives: 8
Normal -> PCOS Overdiagnosis:               0
PCOS -> Normal Underdiagnosis:              0

Total Validation Misclassifications: 22 / 85
patient_id        true_label        pred_label  pcos_prob  df_prob  normal_prob
    DF_003 Dominant Follicle              PCOS     0.6820   0.2758       0.0422
    DF_074 Dominant Follicle              PCOS     0.9830   0.0152       0.0018
    DF_128 Dominant Follicle              PCOS     0.6026   0.3527       0.0447
    DF_151 Dominant Follicle              PCOS     0.5187   0.4421       0.0392
    DF_178 Dominant Follicle              PCOS     0.6515   0.2523       0.0962
    DF_219 Dominant Follicle              PCOS     0.8074   0.1682       0.0244
    DF_259 Dominant Follicle              PCOS     0.9768   0.0224       0.0009
    DF_286 Dominant Follicle              PCOS     0.9705   0.0275       0.002

In [18]:
# CELL 18: Grad-CAM
try:
    # Resolve target layer object from model
    target_layer = model
    for part in target_layer_name.split("."):
        if part.isdigit():
            target_layer = target_layer[int(part)]
        else:
            target_layer = getattr(target_layer, part)

    cam_engine = GradCAM(model, target_layer)

    # Generate Grad-CAM for representative Normal, PCOS, and Dominant Follicle cases
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    sample_indices = [0, 22, 45]  # Representative samples from validation set

    for ax_idx, s_idx in enumerate(sample_indices):
        tensor_in, true_cls, p_path = val_loader.dataset[s_idx]
        tensor_batch = tensor_in.unsqueeze(0).to(device)
        heatmap = cam_engine.generate(tensor_batch, target_class=true_cls)
        
        # Denormalize image for display
        img_disp = tensor_in.permute(1, 2, 0).cpu().numpy()
        img_disp = (img_disp * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
        img_disp = np.clip(img_disp, 0.0, 1.0)
        
        overlay = overlay_cam_on_image(img_disp, heatmap)
        axes[ax_idx].imshow(overlay)
        axes[ax_idx].set_title(
            f"True: {CLASS_NAMES[true_cls]}\nPred: {CLASS_NAMES[preds[s_idx]]} (Prob: {probs[s_idx, true_cls]:.2f})",
            fontsize=10, fontweight="bold"
        )
        axes[ax_idx].axis("off")

    gradcam_save_path = str(OUTPUT_DIR / "gradcam_validation.png")
    plt.tight_layout()
    plt.savefig(gradcam_save_path, dpi=200)
    plt.close()
    print(f"✓ Saved Grad-CAM visualizations to: {gradcam_save_path}")
except Exception as e:
    print(f"Grad-CAM note for {MODEL_NAME}: {e}")

✓ Saved Grad-CAM visualizations to: results\ResNet50\gradcam_validation.png


In [19]:
# CELL 19: Save results
metrics_json_path = OUTPUT_DIR / "metrics.json"
val_metrics["model_name"] = MODEL_NAME
val_metrics["best_epoch"] = best_epoch
val_metrics["checkpoint_path"] = CHECKPOINT_PATH
val_metrics["learning_rate"] = LEARNING_RATE
val_metrics["batch_size"] = BATCH_SIZE

with open(metrics_json_path, "w") as f:
    json.dump(val_metrics, f, indent=2)
print(f"✓ Saved metrics JSON to: {metrics_json_path}")

val_preds_csv_path = OUTPUT_DIR / "validation_predictions.csv"
val_df.to_csv(val_preds_csv_path, index=False)
print(f"✓ Saved validation predictions to: {val_preds_csv_path}")

if history:
    history_csv_path = OUTPUT_DIR / "history.csv"
    pd.DataFrame(history).to_csv(history_csv_path, index=False)
    print(f"✓ Saved training history to: {history_csv_path}")

print(f"\n[COMPLETE] Benchmark execution for {MODEL_NAME} finished successfully!")

✓ Saved metrics JSON to: results\ResNet50\metrics.json
✓ Saved validation predictions to: results\ResNet50\validation_predictions.csv
✓ Saved training history to: results\ResNet50\history.csv

[COMPLETE] Benchmark execution for ResNet50 finished successfully!
